# FSP source adapter

Control panel for `fsp.run` (`1_scrapers/fsp/src/run.py`). All logic lives in the module; this notebook only calls it. Every path comes from `core.paths`. fsp.de needs no key and no login; plain HTTP works (no Cloudflare block seen on 2026-10-06).

**What FSP is.** The TÜV Rheinland partner network. The locator at `www.fsp.de/standorte` is server-rendered HTML (no JSON API): 12 result cards per page, the pagination bar states the page count, no coordinates and no operator on the cards.

**Scope rule (Nikos, 2026-10-06).** A site that offers HU counts, whatever its badge. Gutachtenbüros without HU do not count. "Offers HU" means the site is in the HU-filtered list (`d=hauptuntersuchung`), or, for a site the locator does not list but the sitemap does, its own detail page names Hauptuntersuchung. Doubts are only flagged in `doubt_flags` / `needs_review`; nothing is excluded.

**robots.txt.** fsp.de disallows the `d=` filter. Nikos decided on 2026-10-06 to disregard that. The code still needs `ALLOW_ROBOTS_DISALLOWED = True` (section 1) on every run, so it never happens by accident.

**What a run does.** HU-filtered list (14 pages) → all-services list (27 pages, to see what is *not* HU) → `sitemap.xml` cross-check → detail page of every sitemap-only site. **PASS** needs every check in the summary to hold: page counts equal the pagination bar, the loop ended on a real empty page, no duplicates, HU list inside the all-services list, every listed site in the sitemap, every sitemap-only page readable. A run that is not a PASS labels its CSV `unverified`. Coverage is **not proven** by a PASS: a site missing from both the list and the sitemap cannot be seen.

| Mode | Requests | Goes to |
|---|---|---|
| `simulated` | none (fake fsp.de with known truth, must find 31 of 31) | `test_output/` |
| `sample` | live, 3 (robots.txt + 2 HU pages), reports SAMPLE | `test_output/` |
| `full` | live, ~45, ~3 min, needs `confirm_full_run=True` | `output/` |
| `rebuild` | none, re-parses the raw files of earlier run folders | `output/` |

A run writes only `fsp_pruefstellen_<date>.csv` (the in-scope rows), `run_summary.json`/`.md`, `query_log.csv` and, if `SAVE_RAW`, `raw/` (list pages gzipped, a few MB). The files contain contact data and stay out of git.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from core import paths
from fsp import run as fsp

ROOT = paths.project_root()
SAVE_RAW = True  # list pages are stored gzipped; without raw a run cannot be re-parsed offline later
ALLOW_ROBOTS_DISALLOWED = True  # Nikos, 2026-10-06: disregard robots.txt for the HU filter (d=)

# Raw files of the live recon of 2026-10-06 (all-services list + sitemap, and the HU-filtered list)
RECON_RUNS = [
    paths.scraper_dir("fsp") / "test_output" / "2026-10-06_0954_recon",
    paths.scraper_dir("fsp") / "test_output" / "2026-10-06_1022_recon",
]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
ROOT

## 2. Offline: re-parse the saved recon (no requests)

Runs the whole pipeline (parser, scope rule, checks) on the raw HTML saved by the live recon of 2026-10-06 and writes a `rebuild` run folder in `1_scrapers/fsp/output/`. Each execution makes a new folder; nothing is overwritten.

In [ ]:
rb = fsp.run(mode="rebuild", source_runs=RECON_RUNS, allow_robots_disallowed=ALLOW_ROBOTS_DISALLOWED)
print(rb.outcome, "-", len(rb.frame), "sites in scope")
print(rb.run_dir.relative_to(ROOT))
pd.DataFrame(rb.summary["checks"])

## 3. Inspect

In [ ]:
df = rb.frame
print("Counts:", rb.summary["counts"])
print()
print("Why in scope:", dict(df["scope_reason"].value_counts()))
print("Badges:", dict(df["type_badges"].value_counts()))
print()
print("Doubt flags (a row can carry several):")
df["doubt_flags"].replace("", pd.NA).dropna().str.split("; ").explode().value_counts()

In [ ]:
# Sitemap-only sites: pages the sitemap lists but the locator does not
pd.DataFrame(rb.summary["sitemap_only"])

In [ ]:
review = df[df["needs_review"]]
print(f"{len(review)} rows are flagged for review (they stay in the export)")
review[["source_id", "name", "plz", "city", "type_badges", "scope_reason", "doubt_flags"]]

In [ ]:
# PLZ areas (first digit) as a rough regional spread; FSP gives no Bundesland
df["plz"].str[:1].value_counts().sort_index()

In [ ]:
# Five random "Prüfstelle ..." sites to look up on Google Maps (clean addresses only)
pool = df[df["name"].str.match(r"(Kfz-)?Prüfstelle\b") & (df["doubt_flags"] == "")]
pool.sample(5, random_state=1)[["name", "street", "plz", "city"]]

## 4. Fetch

Sections 4a–4c write one run folder `YYYY-MM-DD_HHMM_<mode>`. The cells are self-contained: they run right after a kernel restart without the cells above.

### 4a. Simulated run (no requests)

A fake fsp.de with known truth (30 HU sites, 1 Prüfstelle-badged Sachverständigenbüro outside the HU list, 8 Gutachtenbüros, 2 sitemap-only pages of which one names HU). Must PASS with 31 sites.

In [ ]:
from core import paths
from fsp import run as fsp

sim = fsp.run(mode="simulated", allow_robots_disallowed=True, save_raw=False)
print(sim.outcome, "-", len(sim.frame), "of 31 expected")
sim.frame[["location_id", "name", "scope_reason", "doubt_flags"]].tail(3)

### 4b. Sample run (live, 3 requests)

**Live requests.** robots.txt plus pages 1 and 2 of the HU-filtered list (24 sites). Confirms that fsp.de still answers with plain HTTP, that `&page=N` still pages, and that the parser still reads the cards. A sample always reports `SAMPLE`: it has no completeness proof. Uses the HU filter, which robots.txt disallows (see the top).

In [ ]:
from core import paths
from fsp import run as fsp

ROOT = paths.project_root()
SAVE_RAW = True
ALLOW_ROBOTS_DISALLOWED = True  # Nikos, 2026-10-06

sample = fsp.run(mode="sample", root=ROOT, allow_robots_disallowed=ALLOW_ROBOTS_DISALLOWED, save_raw=SAVE_RAW)  # 3 live requests
sample.frame[["name", "plz", "city", "type_badges", "doubt_flags"]].head(10)

### 4c. Full run (live, ~45 requests, ~3 minutes)

**Live requests.** Needs `confirm_full_run=True`. The committed default is `False`: with it the cell stops with an error on purpose before any request is sent. Pace: 2-4 s random pause, one request at a time, at most 3 retries with backoff; any Cloudflare page, 403/503 or result page without cards and without the results heading stops the run (never parsed as "0 results").

Expect about 168 sites from the HU list, 313 in the all-services list, 315 in the sitemap and 1 sitemap-only site with HU (169 in scope on 2026-10-06), but the figures come from the run: compare them with the summary, not with this text. A drop of more than 10 % against the last passing full run is reported as a warning in the summary.

In [ ]:
from core import paths
from fsp import run as fsp

ROOT = paths.project_root()
SAVE_RAW = True
ALLOW_ROBOTS_DISALLOWED = True  # Nikos, 2026-10-06

# Set confirm_full_run=True to start the full run. Keep False in the committed notebook.
full = fsp.run(mode="full", confirm_full_run=False, root=ROOT,
               allow_robots_disallowed=ALLOW_ROBOTS_DISALLOWED, save_raw=SAVE_RAW)

In [ ]:
# After a full run: outcome, checks and warnings
print(full.outcome, "-", len(full.frame), "sites in scope")
print(full.summary["warnings"] or "no warnings", "| drop check:", full.summary["drop_check"])
pd.DataFrame(full.summary["checks"])

## 5. Compare two runs (change detection)

Which sites appeared or disappeared between two runs of `output/`. Set `OLD` and `NEW` to run-folder names.

In [ ]:
import pandas as pd
from core import paths

out = paths.output_dir("fsp", "full")
runs = sorted(p.name for p in out.glob("*_full") if (p / "run_summary.json").exists()) + \
       sorted(p.name for p in out.glob("*_rebuild"))
print("Runs in output/:", runs)

OLD, NEW = None, None  # e.g. "2026-10-06_1025_rebuild", "2026-11-03_0900_full"
if OLD and NEW:
    def ids(name):
        f = next((out / name).glob("fsp_pruefstellen_*.csv"))
        return set(pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str)["source_id"])
    old, new = ids(OLD), ids(NEW)
    print(f"{len(old)} -> {len(new)} | new: {sorted(new - old)} | gone: {sorted(old - new)}")